# Modelo de Predicción de ROI — Procesos RPA

**Objetivo:** Entrenar un modelo de machine learning que prediga el ROI (%) de una automatización RPA antes de implementarla, a partir de características observables del proceso.

**Flujo del notebook:**
1. Carga del dataset de ROI calculado
2. Análisis de features y target
3. Comparación de algoritmos (Linear, RF, GBM, XGBoost)
4. Entrenamiento y evaluación del mejor modelo
5. Importancia de variables
6. Guardado del modelo para producción

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost no disponible — se omitirá en la comparación.')

from src.utils.roi_calculator import build_roi_dataset
from src.models.roi_predictor import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGET,
    prepare_features, train, get_feature_importance
)

pd.set_option('display.float_format', '{:,.3f}'.format)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

print('Entorno listo.')

## 1. Carga y preparación del dataset

In [ ]:
df_raw = build_roi_dataset()
df_model = df_raw.dropna(subset=[TARGET, 'TiempoManualHoras', 'ValorHoraPromedio']).copy()

print(f'Filas totales:              {len(df_raw)}')
print(f'Filas con ROI calculable:   {len(df_model)}')
print(f'Target — ROI_Porcentaje:')
print(df_model[TARGET].describe().to_string())

In [ ]:
# Distribución del target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_model[TARGET], bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución ROI (%) — target')
axes[0].set_xlabel('ROI (%)')

# Log-transform
roi_pos = df_model[df_model[TARGET] > 0][TARGET]
axes[1].hist(np.log1p(roi_pos), bins=15, color='seagreen', edgecolor='white')
axes[1].set_title('Distribución log(1 + ROI) — bots con ROI > 0')
axes[1].set_xlabel('log(1 + ROI)')

plt.tight_layout()
plt.show()

In [ ]:
X = prepare_features(df_model)
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} muestras | Test: {len(X_test)} muestras')
print(f'Features numéricas:    {len(NUMERIC_FEATURES)}')
print(f'Features categóricas:  {len(CATEGORICAL_FEATURES)}')
print(f'\nValores nulos en X_train: {X_train.isnull().sum().sum()}')

## 2. Comparación de algoritmos

In [ ]:
def make_pipeline(estimator):
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
    ])
    return Pipeline([('pre', preprocessor), ('model', estimator)])

candidates = {
    'Ridge':            make_pipeline(Ridge(alpha=10)),
    'Lasso':            make_pipeline(Lasso(alpha=1)),
    'RandomForest':     make_pipeline(RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    'GradientBoosting': make_pipeline(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)),
}

if XGBOOST_AVAILABLE:
    candidates['XGBoost'] = make_pipeline(XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                                        random_state=42, verbosity=0, device='gpu'))

kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, pipe in candidates.items():
    cv_r2 = cross_val_score(pipe, X, y, cv=kf, scoring='r2')
    cv_mae = -cross_val_score(pipe, X, y, cv=kf, scoring='neg_mean_absolute_error')
    results.append({
        'Modelo': name,
        'CV R² (media)': cv_r2.mean(),
        'CV R² (std)': cv_r2.std(),
        'CV MAE (media)': cv_mae.mean(),
    })
    print(f'{name:<20} R²={cv_r2.mean():.3f} ± {cv_r2.std():.3f}  |  MAE={cv_mae.mean():.1f}%')

df_results = pd.DataFrame(results).sort_values('CV R² (media)', ascending=False)
df_results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(df_results))]
ax.barh(df_results['Modelo'], df_results['CV R² (media)'], xerr=df_results['CV R² (std)'],
        color=colors, capsize=4, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Comparación de modelos — R² (CV 5-fold)')
ax.set_xlabel('R² promedio')
plt.tight_layout()
plt.show()

best_model_name = df_results.iloc[0]['Modelo']
print(f'\nMejor modelo: {best_model_name}')

## 3. Entrenamiento y evaluación del mejor modelo

In [ ]:
metrics = train(df_model)

print('=== Métricas del modelo final (GradientBoosting) ===')
print(f"  R²  (test):          {metrics['r2']:.4f}")
print(f"  MAE (test):          {metrics['mae']:.2f}%")
print(f"  RMSE(test):          {metrics['rmse']:.2f}%")
print(f"  R²  (CV 5-fold):     {metrics['cv_r2_mean']:.4f} ± {metrics['cv_r2_std']:.4f}")
print(f"  Muestras train/test: {metrics['n_train']} / {metrics['n_test']}")

In [ ]:
# Predicciones en test para visualizar ajuste
from src.models.roi_predictor import load_model

pipeline = load_model()
X_test_prepared = prepare_features(df_model.loc[X_test.index])
y_pred = pipeline.predict(X_test_prepared)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Predicho vs real
axes[0].scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', s=60)
lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lim, lim, 'r--', label='Predicción perfecta')
axes[0].set_xlabel('ROI real (%)')
axes[0].set_ylabel('ROI predicho (%)')
axes[0].set_title(f'Predicho vs Real  (R²={metrics["r2"]:.3f})')
axes[0].legend()

# Residuos
residuos = y_pred - y_test.values
axes[1].hist(residuos, bins=15, color='salmon', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Distribución de residuos')
axes[1].set_xlabel('Residuo = predicho − real (%)')

plt.tight_layout()
plt.savefig('../reports/figures/modelo_evaluacion.png', bbox_inches='tight')
plt.show()

## 4. Importancia de variables

In [ ]:
df_imp = get_feature_importance(top_n=15)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2ecc71' if i < 5 else '#3498db' for i in range(len(df_imp))]
ax.barh(df_imp['feature'][::-1], df_imp['importance'][::-1], color=colors[::-1], edgecolor='white')
ax.set_title('Importancia de variables — GradientBoosting')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', bbox_inches='tight')
plt.show()

print('\nTop 5 factores que más explican el ROI:')
for _, row in df_imp.head(5).iterrows():
    print(f"  {row['feature']:<35} {row['importance']:.4f}")

## 5. Predicciones para todos los bots del portafolio

In [ ]:
X_all = prepare_features(df_model)
df_model['ROI_Predicho'] = pipeline.predict(X_all)
df_model['Error_Prediccion'] = df_model['ROI_Predicho'] - df_model['ROI_Porcentaje']

comparacion = df_model[['Automatizacion','ROI_Porcentaje','ROI_Predicho','Error_Prediccion']].copy()
comparacion = comparacion.sort_values('ROI_Porcentaje', ascending=False)
comparacion['ROI_Porcentaje'] = comparacion['ROI_Porcentaje'].round(1)
comparacion['ROI_Predicho'] = comparacion['ROI_Predicho'].round(1)
comparacion['Error_Prediccion'] = comparacion['Error_Prediccion'].round(1)

print('Predicción vs. ROI real (todos los bots):')
comparacion

## 6. Ejemplo de predicción para un nuevo bot

In [ ]:
from src.models.roi_predictor import predict

nuevo_bot = {
    'TiempoManualHoras':     3.0,     # 3 horas de trabajo manual por ejecución
    'Num_Ejecuciones':       200,     # 200 ejecuciones esperadas
    'ValorHoraPromedio':     35000,   # $35,000 COP/hora
    'Tecnologia':            'UiPath',
    'Estado':                'Activo',
    'DuracionPromedio_Horas': 0.25,   # Robot tarda 15 minutos
    'PromTransacciones':     10,
    'TasaExito':             0.95,
    'TasaError':             0.02,
    'EjecucionesPorDia':     200/365,
    'DiasEnProduccion':      365,
    'NumAreas':              2,
    'NumRoles':              1,
}

resultado = predict(nuevo_bot)

print('=== Predicción para nuevo bot ===')
print(f"  ROI predicho:         {resultado['roi_porcentaje']:.0f}%")
print(f"  Ahorro neto:          ${resultado['ahorro_neto_cop']/1e6:.2f}M COP")
print(f"  Beneficio bruto:      ${resultado['beneficio_bruto_cop']/1e6:.2f}M COP")
print(f"  Costo robot:          ${resultado['costo_robot_cop']/1e6:.2f}M COP")

if resultado['roi_porcentaje'] > 500:
    print('\nVeredicto: EXCELENTE candidato para automatización.')
elif resultado['roi_porcentaje'] > 100:
    print('\nVeredicto: BUENA candidata para automatización.')
else:
    print('\nVeredicto: Evaluar con cuidado antes de automatizar.')

In [ ]:
print('Modelo entrenado y guardado en: models/roi_model.joblib')
print('Listo para usar desde la aplicación Streamlit (app/chat.py)')